# Interactive RDR Learning
## Last updated: 12 April 2026

Single Classification Ripple Down Rules (SCRDR) ရဲ့ အလုပ်လုပ်ပုံကို နားလည်နိုင်ဖို့အတွက် လုပ်ပေးတဲ့ tutorial ပါ။ ပထမဆုံး CSV နဲ့ ဆောက်ထားတဲ့ example dataset ကို လေ့လာပြီး classification result ကောင်းဖို့အတွက် rule တွေကို ဆောက်ကြီးပြီး run ကြည့်ပါ။  

In [1]:
%cd ./inter

/home/ye/exp/scrdr_tuto/inter


In [2]:
!ls *.py

scrdr_interactive.py


In [3]:
!cat scrdr_interactive.py

## For understanding SCRDR to my AI Engineering (Fundamental) Class students
## Written by Ye Kyaw Thu, LU Lab., Myanmar
## Last Updated: 12 April 2026
##
## Reference: 
## The Alternative to Machine Learning
## by Paul Compton (The University of NSW, Sydney, Australia), 
## Byeong Ho Kang (The University of Tasmania, Hobart, Australia)


import pandas as pd
import json
import argparse
import os
import sys
from sklearn.metrics import classification_report

class RDRNode:
    def __init__(self, condition=None, conclusion=None):
        self.condition = condition # {'col':.., 'op':.., 'val':..}
        self.conclusion = str(conclusion) if conclusion is not None else None
        self.if_true = None   # Refinement (Exception)
        self.if_false = None  # Alternative

    def evaluate(self, row):
        # The root/default node always matches 
        if not self.condition or not self.condition['col']:
            return True
        col, op, val = self.condition['col'], self.condition[

## Stopping Criteria

Python code ဟာ Single Classification Ripple Down Rules (SCRDR) ရဲ့ အခြေခံ သဘောတရားဖြစ်တဲ့ "Knowledge is acquired in context" (အခြေအနေအရ သိရှိနားလည်မှုကို ရယူခြင်း) ဆိုတဲ့ အချက်အပေါ် အခြေခံပြီး တည်ဆောက်ထားတာ ဖြစ်ပါတယ်။  

ဒီ program က Rule အသစ်တွေ တောင်းတာကို အောက်ပါ အခြေအနေ (၃) ခုမှာ ရပ်တန့်သွားမှာ ဖြစ်ပါတယ်။  

**Correct Prediction:** `main()` function ထဲက loop မှာ `if args.mode == 'build' and predicted != actual:` ဆိုတဲ့ condition ပါပါတယ်။ ဆိုလိုတာက လက်ရှိ Tree ထဲမှာရှိတဲ့ Rule တွေနဲ့ စစ်လိုက်လို့ အဖြေမှန် (Actual) ထွက်နေတယ်ဆိုရင် Rule ထပ်မတောင်းဘဲ နောက် row တစ်ခုကို အလိုအလျောက် ဆက်သွားပါတယ်။

**End of Dataset:** `for idx, row in df.iterrows():` loop ပြီးသွားရင် (CSV ထဲက အချက်အလက်အားလုံးကို စစ်ဆေးပြီးသွားရင်) အလိုအလျောက် ရပ်သွားပါတယ်။  

**Manual Exist:** `get_validated_input` ထဲမှာ 'exit' လို့ ရိုက်လိုက်ရင် `add_rule` က `False` ပြန်ပေးပြီး `main` ထဲမှာ `break` နဲ့ loop ကို ထွက်သွားစေပါတယ်။  


## SCRDR Rule Building with Students Dataset


In [4]:
!ls ./data/

loan_approval.csv  students.csv  weather_activity.csv


In [6]:
!cat ./data/students.csv

Student_ID,Hours_Studied,Attendance_Pct,Pass_Fail
1,2,40,Fail
2,15,95,Pass
3,12,80,Pass
4,1,30,Fail
5,8,45,Fail
6,20,90,Pass
7,7,85,Pass
8,3,50,Fail
9,18,88,Pass
10,5,92,Fail


In [7]:
%pwd

'/home/ye/exp/scrdr_tuto/inter'

In [8]:
!python ./scrdr_interactive.py --help

usage: scrdr_interactive.py [-h] --input INPUT --target TARGET [--tree TREE]
                            [--exclude [EXCLUDE ...]] [--mode {build,test}]

Interactive RDR Knowledge Builder

options:
  -h, --help            show this help message and exit
  --input INPUT         Dataset CSV file
  --target TARGET       Target column to predict
  --tree TREE           Output JSON model file
  --exclude [EXCLUDE ...]
                        Columns to ignore (IDs, Names)
  --mode {build,test}   build: Interactive training | test: Run model on data

Interactive Workflow:
1. Use --mode build to teach the model. The script only asks for rules when it is wrong.
2. Type 'exit' at any prompt to save the current tree and stop.
3. Use --mode test to evaluate a built model without being prompted for rules.
    


In [26]:
!cat ./run_inter_students.sh

#!/bin/bash

## Building RDR Model
python scrdr_interactive.py --input ./data/students.csv --target Pass_Fail \
--exclude Student_ID --tree students_rules_demo.json --mode build | tee running_inter_students_demo.log

## Testing
python scrdr_interactive.py --input ./data/students.csv --target Pass_Fail \
--exclude Student_ID --tree students_rules_demo.json --mode test


## 1st time running


## Check Built Model

In [10]:
!cat students_rules_demo.json

{
  "condition": null,
  "conclusion": "Unknown",
  "if_true": {
    "condition": {
      "col": "Hours_Studied",
      "op": "<",
      "val": "7"
    },
    "conclusion": "Fail",
    "if_true": null,
    "if_false": {
      "condition": {
        "col": "Attendance_Pct",
        "op": ">",
        "val": "50"
      },
      "conclusion": "Pass",
      "if_true": null,
      "if_false": {
        "condition": {
          "col": "Attendance_Pct",
          "op": "<",
          "val": "50"
        },
        "conclusion": "Fail",
        "if_true": null,
        "if_false": null
      }
    }
  },
  "if_false": null
}

## Testing with Built RDR Model

In [11]:
!python scrdr_interactive.py --input ./data/students.csv --target Pass_Fail \
--exclude Student_ID --tree students_rules_demo.json --mode test

Loaded existing model: students_rules_demo.json
--- Running in TEST mode ---

--- FINAL PERFORMANCE SUMMARY ---
              precision    recall  f1-score   support

        Fail       1.00      1.00      1.00         5
        Pass       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



## 2nd Time Running


In [12]:
!cat 2nd-time-running.log

ye@lst-hpc3090:~/exp/scrdr_tuto/inter$ ./run_inter_students.sh
--- Running in BUILD mode ---

[Row 1/10] ERROR Found!
{'Hours_Studied': 2, 'Attendance_Pct': 40, 'Pass_Fail': 'Fail'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'Fail'
Enter feature name (or 'exit'): Attendance_Pct
Enter operator (==, <, >): <
Enter threshold value: 50

[Row 2/10] ERROR Found!
{'Hours_Studied': 15, 'Attendance_Pct': 95, 'Pass_Fail': 'Pass'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'Pass'
Enter feature name (or 'exit'): Hours_Studied
Enter operator (==, <, >): >
Enter threshold value: 7

[Row 7/10] ERROR Found!
{'Hours_Studied': 7, 'Attendance_Pct': 85, 'Pass_Fail': 'Pass'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'Pass'
Enter feature name (or 'exit'): Attendance_Pct
Enter operator (==, <, >): >
Enter threshold value: 50

[Row 8/10] ERROR Found!
{'Hours_Studied': 3, 'Attendance_Pct': 50, 'Pass_Fail': 'Fail'}

[KNOWLEDGE ACQUISITI

## Let's check model

In [13]:
!cat students_rules_demo.json

{
  "condition": null,
  "conclusion": "Unknown",
  "if_true": {
    "condition": {
      "col": "Attendance_Pct",
      "op": "<",
      "val": "50"
    },
    "conclusion": "Fail",
    "if_true": null,
    "if_false": {
      "condition": {
        "col": "Hours_Studied",
        "op": ">",
        "val": "7"
      },
      "conclusion": "Pass",
      "if_true": null,
      "if_false": {
        "condition": {
          "col": "Attendance_Pct",
          "op": ">",
          "val": "50"
        },
        "conclusion": "Pass",
        "if_true": {
          "condition": {
            "col": "Hours_Studied",
            "op": "<",
            "val": "7"
          },
          "conclusion": "Fail",
          "if_true": null,
          "if_false": null
        },
        "if_false": {
          "condition": {
            "col": "Hours_Studied",
            "op": "<",
            "val": "7"
          },
          "conclusion": "Fail",
          "if_true": null,
          "if_false": null

## Classification Engine (သို့မဟုတ်) Decision  

ယူဇာက ပေးထားတဲ့ target column အတွက် အဖြေတစ်ခုကို ဘယ်လို ဆုံးဖြတ်သလဲဆိုတာကို `SCRDR_Engine.classify` နဲ့ `RDRNode.evaluate` တို့က အဓိက လုပ်ဆောင်ပေးပါတယ်။  

### Logic Flow  

1. **Start at Root:** စစ်ဆေးမှုကို `self.root` (ဒီမှာဆိုရင် 'Unknown' node) ကနေ စပါတယ်။
2. **Evaluate:** `evaluate(row)` function က row ထဲက တန်ဖိုး (ဥပမာ- student dataset မှာ ဆိုရင် `Hours_Studied`) ဟာ သတ်မှတ်ထားတဲ့ Rule (ဥပမာ- `> 7`) နဲ့ ကိုက်ညီလားဆိုတာ စစ်ပါတယ်။
3. **Binary Path:**
    - **Match:** `last_match` ကို အဲ့ဒီ node အဖြစ် မှတ်သားပြီး `if_true` (Refinement/အသေးစိတ်ပြင်ဆင်ချက်) လမ်းကြောင်းအတိုင်း ဆက်သွားပါတယ်။ ဒါဟာ "မှန်တယ်၊ ဒါပေမဲ့ နောက်ထပ် ထူးခြားချက်ရှိသေးလား" လို့ မေးတာနဲ့ တူပါတယ်။
    - **No Match:** `if_false` (Alternative/အခြားရွေးချယ်စရာ) လမ်းကြောင်းအတိုင်း ဆက်သွားပါတယ်။ ဒါဟာ "ဒီ rule နဲ့ မကိုက်ညီရင် နောက် rule တစ်ခုနဲ့ စစ်ကြည့်မယ်" လို့ ဆိုလိုတာပါ။
4. **Final Decision:** နောက်ထပ်သွားစရာ လမ်းကြောင်းမရှိတော့တဲ့အခါ နောက်ဆုံး 'Match' ဖြစ်ခဲ့တဲ့ node ရဲ့ `conclusion` ကို အဖြေအဖြစ် ထုတ်ပေးပါတယ်။

## Walkthrough with Build SCRDR Model

အထက်မှာ ပြထားတဲ့ running log ကိုအခြေခံပြီး ဆရာရေးထားတဲ့ code က ဘယ်လို classification လုပ်သွားသလဲ၊ ဘယ်လို rule အသစ်တောင်းတာလဲ ဆိုတာကို ရှင်းပြပါမယ်။  

```
[Row 10/10] ERROR Found!
{'Hours_Studied': 5, 'Attendance_Pct': 92, 'Pass_Fail': 'Fail'}
```

Training လုပ်နေစဉ်... အထက်က အဆင့်ကနေ ဖြတ်ပြီး ရှင်းပြပါမယ်။   

ဒီအဆင့်မှာ လက်ရှိ system သို့မဟုတ် RDR Tree ရဲ့ prediction က `Pass` ဆိုပြီး ဆုံးဖြတ်တာမို့ မှားပါတယ်။ အဲဒါကြောင့် သူက dataset ထဲက target column ရဲ့ အဖြေနဲ့ တိုက်ကြည့်ပြီး `ERROR Found!` ဆိုပြီး ပြောတာပါ။  

Code ထဲမှာ ရေးထားတဲ့အတိုင်း `predicted ('Pass') != actual ('Fail')` ဖြစ်သွားတဲ့အတွက် logic အရ `add_rule` function ကို ခေါ်လိုက်ပါတယ်။  

ဒီအထိလိုက်လို့ရရင်တော့ ဘယ်လိုအခြေအနေမှာ Rule အသစ်ဖြည့်ဖို့ ယူဇာဆီက တောင်းသလဲ ဆိုတာကို နားလည်ပါလိမ့်မယ်။  


## Decision Making with Build SCRDR Tree or Model  

Training လုပ်ပြီးလို့ ထွက်လာတဲ့ JSON model နဲ့ Testing အဆင့်မှာ Row 10 (10,5,92,Fail) ကို classify လုပ်ကြည့်ရအောင်။  

1. **Root:** Match ဖြစ်တယ် (Default အမြဲမှန်) -> `last_match` က Unknown ဖြစ်သွားမယ်။ `if_true` ကို သွားမယ်။  
2. **`Attendance_Pct < 50:`** ၉၂ ဖြစ်လို့ False (မကိုက်ညီ) -> `if_false` ကို သွားမယ်။  
3. **Hours_Studied > 7:** ၅ ဖြစ်လို့ False (မကိုက်ညီ) -> `if_false` ကို သွားမယ်။  
4. **Attendance_Pct > 50:** ၉၂ ဖြစ်လို့ True (ကိုက်ညီတယ်) -> `last_match` က Pass ဖြစ်သွားပြီ။ အခု `if_true` ကို ဆက်သွားမယ်။  
5. **`Hours_Studied < 7:`** ၅ ဖြစ်လို့ True (ကိုက်ညီတယ်) -> `last_match` က Fail ဖြစ်သွားပြီ။ ရှေ့ဆက်စရာ မရှိတော့ဘူး။  

နောက်ဆုံး classification ရလဒ်အနေနဲ့ system က Fail လို့ အဖြေထုတ်ပေးပါလိမ့်မယ်။  

Note: အထက်က JSON RDR model နဲ့ တိုက်ကြည့်ပါ။  

## 1st Time Build/Test SCRDR with Loan Example Dataset

Exercise အနေနဲ့ရော ပိုလည်းနားလည်အောင်လို့ နောက်ထပ် ဥပမာ dataset အသေးလေးတစ်ခု ဖြစ်တဲ့ loan_approval.csv နဲ့ လေ့လာကြည့်ကြရအောင်။  

In [14]:
!cat ./data/loan_approval.csv

ID,Income,Credit_Score,Stable_Job,Approved
1,5000,750,Yes,Yes
2,2000,600,No,No
3,8000,580,Yes,No
4,4500,720,No,Yes
5,12000,400,Yes,No
6,3000,710,Yes,Yes


In [15]:
!cat ./run_inter_loan.sh

#!/bin/bash

# Building RDR Rules
python scrdr_interactive.py --input ./data/loan_approval.csv --target Approved \
--exclude ID --tree loan_rules_demo.json --mode build | tee running_inter_loan_demo.log

# Testing with RDR Model
python scrdr_interactive.py --input ./data/loan_approval.csv --target Approved \
--exclude ID --tree loan_rules_demo.json --mode test


## 1st Time Running Log with Loan Dataset

In [17]:
!cat 1st-time-run-loan.log

ye@lst-hpc3090:~/exp/scrdr_tuto/inter$ ./run_inter_loan.sh
--- Running in BUILD mode ---

[Row 1/6] ERROR Found!
{'Income': 5000, 'Credit_Score': 750, 'Stable_Job': 'Yes', 'Approved': 'Yes'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'Yes'
Enter feature name (or 'exit'): Credit_Score
Enter operator (==, <, >): >
Enter threshold value: 700

[Row 2/6] ERROR Found!
{'Income': 2000, 'Credit_Score': 600, 'Stable_Job': 'No', 'Approved': 'No'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'No'
Enter feature name (or 'exit'): Income
Enter operator (==, <, >): <
Enter threshold value: 2000

[Row 3/6] ERROR Found!
{'Income': 8000, 'Credit_Score': 580, 'Stable_Job': 'Yes', 'Approved': 'No'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'No'
Enter feature name (or 'exit'): Stable_Job
Enter operator (==, <, >): ==
Enter threshold value: Yes

--- FINAL PERFORMANCE SUMMARY ---
              precision    recall  f1-score   support

 

ကိုယ်ပေးလိုက်တဲ့ rule တွေအပေါ်ကို မူတည်ပြီး classification ရလဒ်က အထက်မှာ မြင်ရတဲ့အတိုင်းပါပဲ။ F1, precision, recall ရလဒ်တွေကို လေ့လာကြည့်ပါ။  

## Let's Check RDR Rules

Output အဖြစ် ဆောက်ပြီးထွက်လာတဲ့ RDR Tree or JSON ဖိုင်က အောက်ပါအတိုင်းပါ။  

In [18]:
!cat loan_rules_demo.json

{
  "condition": null,
  "conclusion": "Unknown",
  "if_true": {
    "condition": {
      "col": "Credit_Score",
      "op": ">",
      "val": "700"
    },
    "conclusion": "Yes",
    "if_true": null,
    "if_false": {
      "condition": {
        "col": "Income",
        "op": "<",
        "val": "2000"
      },
      "conclusion": "No",
      "if_true": null,
      "if_false": {
        "condition": {
          "col": "Stable_Job",
          "op": "==",
          "val": "Yes"
        },
        "conclusion": "No",
        "if_true": null,
        "if_false": null
      }
    }
  },
  "if_false": null
}

## 2nd Time Running with Loan Dataset

loan ဒေတာကိုပဲ သုံးပြီး ဒုတိယအကြိမ် AI Class အတန်းက ကျောင်းသားတွေနဲ့ ထပ် run ထားတဲ့ log ပါ။  

In [19]:
!cat 2nd-time-run-loan-data.log

ye@lst-hpc3090:~/exp/scrdr_tuto/inter$ ./run_inter_loan.sh
Loaded existing model: loan_rules_demo.json
--- Running in BUILD mode ---

[Row 2/6] ERROR Found!
{'Income': 2000, 'Credit_Score': 600, 'Stable_Job': 'No', 'Approved': 'No'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'No'
Enter feature name (or 'exit'): Income
Enter operator (==, <, >): <
Enter threshold value: 3000

--- FINAL PERFORMANCE SUMMARY ---
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         3
         Yes       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

Loaded existing model: loan_rules_demo.json
--- Running in TEST mode ---

--- FINAL PERFORMANCE SUMMARY ---
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         3
         Yes       

## Let's check 2nd time model

ဒီတစ်ခါ အသစ်ဆောက်ပြီးထွက်လာတဲ့ RDR Tree က အောက်ပါအတိုင်းပါ။  

In [20]:
!cat loan_rules_demo.json

{
  "condition": null,
  "conclusion": "Unknown",
  "if_true": {
    "condition": {
      "col": "Credit_Score",
      "op": ">",
      "val": "700"
    },
    "conclusion": "Yes",
    "if_true": null,
    "if_false": {
      "condition": {
        "col": "Income",
        "op": "<",
        "val": "2000"
      },
      "conclusion": "No",
      "if_true": null,
      "if_false": {
        "condition": {
          "col": "Stable_Job",
          "op": "==",
          "val": "Yes"
        },
        "conclusion": "No",
        "if_true": null,
        "if_false": {
          "condition": {
            "col": "Income",
            "op": ">",
            "val": "2999"
          },
          "conclusion": "No",
          "if_true": null,
          "if_false": {
            "condition": {
              "col": "Credit_Score",
              "op": ">",
              "val": "600"
            },
            "conclusion": "No",
            "if_true": null,
            "if_false": {
              "

## Build/Test with Example Weather Dataset

ဒီတစ်ခါတော့ ရာသီဥတု အခြေအနေပေါ်ကို မူတည်ပြီး Activity ကို ခန့်မှန်းပေးတဲ့ classification အလုပ်ကို SCRDR rule တွေဆောက်ပြီး လုပ်ကြည့်ကြရအောင်။  

SCRDR နဲ့ လုပ်မယ်ဆိုရင် ထုံးစံအတိုင်းပဲ ဒေတာကို အရင်ဆုံး နားလည်အောင် လေ့လာပါ။ ကိုယ် နားလည်ရင် နားလည်သလို rule သတ်မှတ်တဲ့နေရာမှာ ပိုပြီးတော့ efficient ဖြစ်တဲ့ SCRDR tree ကို ဆောက်နိုင်ပါလိမ့်မယ်။  

ဒေတာအရမ်းများလာရင်တော့ domain expert နဲ့ တိုင်ပင်ဆွေးနွေးရပါလိမ့်မယ်။  
ကျပန်း ဒေတာတချို့ကို ဆွဲထုတ်ပြီး rule တွေကို ထုတ်ကြည့်တာမျိုး လုပ်ရပါလိမ့်မယ်။  

In [21]:
!cat ./data/weather_activity.csv

Day,Outlook,Temp,Humidity,Windy,Activity
1,Sunny,Hot,High,False,Swim
2,Sunny,Hot,High,True,Stay-In
3,Overcast,Hot,High,False,Swim
4,Rain,Mild,High,False,Walk
5,Rain,Cool,Normal,False,Walk
6,Rain,Cool,Normal,True,Stay-In
7,Overcast,Cool,Normal,True,Walk


## Check Shell Code

Weather ဒေတာနဲ့ run ဖို့အတွက် ပြင်ဆင်ထားတဲ့ shell script က အောက်ပါအတိုင်းပါ။  

In [23]:
!cat ./run_inter_weather.sh

#!/bin/bash

# Building RDR Rules
python scrdr_interactive.py --input ./data/weather_activity.csv --target Activity \
--exclude Day --tree weather_demo.json --mode build | tee running_inter_weather_demo.log

# Testing with RDR Model
python scrdr_interactive.py --input ./data/weather_activity.csv --target Activity \
--exclude Day --tree weather_demo.json --mode test


## Running Log

ကျောင်းသားတွေနဲ့ interactive စမ်းထားတဲ့ log က အောက်ပါအတိုင်းပါ။  

In [24]:
!cat 1st-time-run-weather.log

ye@lst-hpc3090:~/exp/scrdr_tuto/inter$ ./run_inter_weather.sh
--- Running in BUILD mode ---

[Row 1/7] ERROR Found!
{'Outlook': 'Sunny', 'Temp': 'Hot', 'Humidity': 'High', 'Windy': False, 'Activity': 'Swim'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'Swim'
Enter feature name (or 'exit'): Windy
Enter operator (==, <, >): ==
Enter threshold value: False

[Row 2/7] ERROR Found!
{'Outlook': 'Sunny', 'Temp': 'Hot', 'Humidity': 'High', 'Windy': True, 'Activity': 'Stay-In'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Unknown' but Actual is: 'Stay-In'
Enter feature name (or 'exit'): Temp
Enter operator (==, <, >): ==
Enter threshold value: Hot

[Row 4/7] ERROR Found!
{'Outlook': 'Rain', 'Temp': 'Mild', 'Humidity': 'High', 'Windy': False, 'Activity': 'Walk'}

[KNOWLEDGE ACQUISITION]
System predicted: 'Swim' but Actual is: 'Walk'
Enter feature name (or 'exit'): Temp
Enter operator (==, <, >): ==
Enter threshold value: Mild

[Row 5/7] ERROR Found!
{'Outlook': 'Rain', 'Te

ရလဒ်အနေနဲ့ အထက်မှာ မြင်ရတဲ့အတိုင်း perfect ပါပဲ။ :)  

## Let's check Model

ပရိုဂရမ်ကို run ပြီးထွက်လာတဲ့ SCRDR မော်ဒယ်ဖိုင်ကို လေ့လာကြည့်ကြရအောင်။  

In [25]:
!cat weather_demo.json

{
  "condition": null,
  "conclusion": "Unknown",
  "if_true": {
    "condition": {
      "col": "Windy",
      "op": "==",
      "val": "False"
    },
    "conclusion": "Swim",
    "if_true": {
      "condition": {
        "col": "Temp",
        "op": "==",
        "val": "Mild"
      },
      "conclusion": "Walk",
      "if_true": null,
      "if_false": {
        "condition": {
          "col": "Temp",
          "op": "==",
          "val": "Cool"
        },
        "conclusion": "Walk",
        "if_true": null,
        "if_false": null
      }
    },
    "if_false": {
      "condition": {
        "col": "Temp",
        "op": "==",
        "val": "Hot"
      },
      "conclusion": "Stay-In",
      "if_true": null,
      "if_false": {
        "condition": {
          "col": "Outlook",
          "op": "==",
          "val": "Rain"
        },
        "conclusion": "Stay-In",
        "if_true": null,
        "if_false": {
          "condition": {
            "col": "Temp",
            "

## To Do or Assignment-4

အဓိက ကတော့ လက်တွေ့ အနီးအနားက လက်လှမ်းမီတဲ့ ပြဿနာ နှစ်ခု ကို dataset အသေးလေး ပြင်လိုက်ပြီး RDR model ကို interactive ဆောက်ကြည့်ပါ။ မော်ဒယ်နှစ်ခုနဲ့ ရလဒ်တွေကို report တင်ပေးပါ။

ဒီ Assignment ကတော့ Group Work ပါ။  
Group ၆ခု ကတော့ Assignment-1 အတွက် ဖွဲ့ပေးတုန်းကအတိုင်းပဲ သွားကြရအောင်။  
ဖြစ်နိုင်ရင် နောက်တပတ်အတွင်း တင်နိုင်ရင် တင်ကြပါ။  